In [1]:
import re

import numpy as np
import pandas as pd
# import dask.dataframe as pd
import sqlite3

from sklearn.preprocessing import StandardScaler

#import xgboost

con = sqlite3.connect('../Dengue20X_timeseries_CPA_NoiseReduction.db')

cursor = con.cursor()
cursor.execute('SELECT name FROM sqlite_master WHERE type="table";')
print(cursor.fetchall())

[('MyExpt_Per_Object',), ('MyExpt_Per_Image',), ('Experiment',), ('sqlite_sequence',), ('Experiment_Properties',), ('MyExpt_Per_Experiment',), ('MyExpt_Per_RelationshipTypes',), ('MyExpt_Per_Relationships',), ('NS4B_Epro_scored_imagelevel',)]


In [2]:
meta = pd.read_sql_query('SELECT ImageNumber, Image_Metadata_WellID, Image_Metadata_PlateID from MyExpt_Per_Image', con)
meta.head()

,ImageNumber,Image_Metadata_WellID,Image_Metadata_PlateID
0,1,A01,TimeSeries_20221028_164132
1,2,A01,TimeSeries_20221028_164132
2,3,A01,TimeSeries_20221028_164132
3,4,A01,TimeSeries_20221028_164132
4,5,A01,TimeSeries_20221028_164132


In [3]:
def _letter_range(start, stop="{", step=1):
    """Yield a range of lowercase letters.""" 
    for ord_ in range(ord(start.upper()), ord(stop.upper()), step):
        yield chr(ord_)

PC, NC = [],[]
for l in list(_letter_range('A', 'Q')):
    for i in range(1, 5):
        NC.append(f"{l}{str(i).zfill(2)}")
    for j in range(5, 9):
        PC.append(f"{l}{str(j).zfill(2)}")

print(NC)
print()
print(PC)

['A01', 'A02', 'A03', 'A04', 'B01', 'B02', 'B03', 'B04', 'C01', 'C02', 'C03', 'C04', 'D01', 'D02', 'D03', 'D04', 'E01', 'E02', 'E03', 'E04', 'F01', 'F02', 'F03', 'F04', 'G01', 'G02', 'G03', 'G04', 'H01', 'H02', 'H03', 'H04', 'I01', 'I02', 'I03', 'I04', 'J01', 'J02', 'J03', 'J04', 'K01', 'K02', 'K03', 'K04', 'L01', 'L02', 'L03', 'L04', 'M01', 'M02', 'M03', 'M04', 'N01', 'N02', 'N03', 'N04', 'O01', 'O02', 'O03', 'O04', 'P01', 'P02', 'P03', 'P04']

['A05', 'A06', 'A07', 'A08', 'B05', 'B06', 'B07', 'B08', 'C05', 'C06', 'C07', 'C08', 'D05', 'D06', 'D07', 'D08', 'E05', 'E06', 'E07', 'E08', 'F05', 'F06', 'F07', 'F08', 'G05', 'G06', 'G07', 'G08', 'H05', 'H06', 'H07', 'H08', 'I05', 'I06', 'I07', 'I08', 'J05', 'J06', 'J07', 'J08', 'K05', 'K06', 'K07', 'K08', 'L05', 'L06', 'L07', 'L08', 'M05', 'M06', 'M07', 'M08', 'N05', 'N06', 'N07', 'N08', 'O05', 'O06', 'O07', 'O08', 'P05', 'P06', 'P07', 'P08']


In [4]:
highCount = pd.read_csv('WellswHighCellCount.csv')
highCountList = highCount['Image_Metadata_WellID'].unique().tolist()
highCountList

['A01',
 'A02',
 'A03',
 'B01',
 'B02',
 'B03',
 'B04',
 'C01',
 'C02',
 'C03',
 'C04',
 'D01',
 'D02',
 'D03',
 'D04',
 'E01',
 'E02',
 'E03',
 'E04',
 'F01',
 'F02',
 'F03',
 'F04',
 'G01',
 'G04',
 'H01',
 'H03',
 'H04',
 'I04',
 'J01',
 'J02',
 'K01',
 'L03',
 'L04',
 'M01',
 'M02',
 'M04',
 'N01',
 'N02',
 'N04',
 'O01',
 'O04',
 'P01',
 'P03',
 'P04',
 'A21',
 'A22',
 'A23',
 'A24',
 'B21',
 'B22',
 'B23',
 'C22',
 'C23',
 'D21',
 'D22',
 'D23',
 'D24',
 'E21',
 'E22',
 'E23',
 'E24',
 'F21',
 'F23',
 'G21',
 'G22',
 'G23',
 'G24',
 'H21',
 'H22',
 'H23',
 'I21',
 'I22',
 'I23',
 'I24',
 'J21',
 'J22',
 'J23',
 'J24',
 'K21',
 'K22',
 'K23',
 'K24',
 'L21',
 'L22',
 'L23',
 'L24',
 'M21',
 'M22',
 'M23',
 'M24',
 'N21',
 'N22',
 'N23',
 'N24',
 'O21',
 'O22',
 'O23',
 'A17',
 'A19',
 'B17',
 'B19',
 'C17',
 'G17',
 'G19',
 'H20',
 'I17',
 'I18',
 'I19',
 'I20',
 'J18',
 'K17',
 'K18',
 'K20',
 'L18',
 'L19',
 'M17',
 'M20',
 'N17',
 'O18',
 'O19',
 'O20',
 'P17',
 'P20',
 'A14',


In [5]:
#select only PC and NC with high cell count to train
PC_highCC = set(PC).intersection(set(highCountList))
NC_highCC = set(NC).intersection(set(highCountList))
print(PC_highCC)
print()
print(NC_highCC)

{'C05', 'I05', 'A08', 'K05', 'B06', 'O06', 'A05', 'H05', 'N07', 'J07', 'P06', 'N08', 'F05', 'J06', 'I07', 'N05', 'I08', 'C06', 'E06', 'J05', 'L05', 'L06', 'E05', 'O05', 'D06', 'B07', 'G05', 'O08', 'F06', 'M05', 'M07', 'A06', 'N06', 'P05', 'C07', 'D05', 'G08', 'E08', 'G06', 'B05', 'H06', 'H07', 'I06', 'L07', 'H08'}

{'E02', 'F04', 'D03', 'H03', 'C04', 'P03', 'K01', 'F03', 'H01', 'H04', 'C03', 'M01', 'M02', 'P01', 'E01', 'O04', 'N04', 'P04', 'D01', 'D02', 'I04', 'B02', 'G04', 'A02', 'E04', 'B01', 'F02', 'C01', 'M04', 'A01', 'A03', 'F01', 'N01', 'D04', 'N02', 'L03', 'G01', 'E03', 'B04', 'J01', 'B03', 'O01', 'C02', 'L04', 'J02'}


In [6]:
PC = list(PC_highCC)
PC.sort()
print(PC)
NC = list(NC_highCC)
NC.sort()
print(NC)

['A05', 'A06', 'A08', 'B05', 'B06', 'B07', 'C05', 'C06', 'C07', 'D05', 'D06', 'E05', 'E06', 'E08', 'F05', 'F06', 'G05', 'G06', 'G08', 'H05', 'H06', 'H07', 'H08', 'I05', 'I06', 'I07', 'I08', 'J05', 'J06', 'J07', 'K05', 'L05', 'L06', 'L07', 'M05', 'M07', 'N05', 'N06', 'N07', 'N08', 'O05', 'O06', 'O08', 'P05', 'P06']
['A01', 'A02', 'A03', 'B01', 'B02', 'B03', 'B04', 'C01', 'C02', 'C03', 'C04', 'D01', 'D02', 'D03', 'D04', 'E01', 'E02', 'E03', 'E04', 'F01', 'F02', 'F03', 'F04', 'G01', 'G04', 'H01', 'H03', 'H04', 'I04', 'J01', 'J02', 'K01', 'L03', 'L04', 'M01', 'M02', 'M04', 'N01', 'N02', 'N04', 'O01', 'O04', 'P01', 'P03', 'P04']


In [7]:
meta = meta.loc[meta['Image_Metadata_WellID'].isin(NC+PC)]
wells  = meta['ImageNumber'].unique().tolist()
meta.head()

,ImageNumber,Image_Metadata_WellID,Image_Metadata_PlateID
0,1,A01,TimeSeries_20221028_164132
1,2,A01,TimeSeries_20221028_164132
2,3,A01,TimeSeries_20221028_164132
3,4,A01,TimeSeries_20221028_164132
4,5,A01,TimeSeries_20221028_164132


In [8]:
query ="SELECT * FROM MyExpt_Per_Object WHERE ImageNumber IN("

In [9]:
query+=f"{wells[0]}"
for w in wells[1:]:
    query+=","
    query+=f"{w}"
query+=")"

In [10]:
data = pd.read_sql_query(query, con)
data.head()

,ImageNumber,ObjectNumber,Cells_Number_Object_Number,Cells_AreaShape_Area,Cells_AreaShape_BoundingBoxArea,Cells_AreaShape_BoundingBoxMaximum_X,Cells_AreaShape_BoundingBoxMaximum_Y,Cells_AreaShape_BoundingBoxMinimum_X,Cells_AreaShape_BoundingBoxMinimum_Y,Cells_AreaShape_Center_X,...,Nuclei_Texture_Variance_EproAfterMath_6_02_256,Nuclei_Texture_Variance_EproAfterMath_6_03_256,Nuclei_Texture_Variance_Hoe_6_00_256,Nuclei_Texture_Variance_Hoe_6_01_256,Nuclei_Texture_Variance_Hoe_6_02_256,Nuclei_Texture_Variance_Hoe_6_03_256,Nuclei_Texture_Variance_NS4BAfterMath_6_00_256,Nuclei_Texture_Variance_NS4BAfterMath_6_01_256,Nuclei_Texture_Variance_NS4BAfterMath_6_02_256,Nuclei_Texture_Variance_NS4BAfterMath_6_03_256
0,1,1,1,2565.0,4080.0,622.0,48.0,537.0,0.0,576.887329,...,0.0,0.0,9.712645,10.273859,9.904125,10.098906,0.0,0.0,0.0,0.0
1,1,2,2,1682.0,2090.0,1179.0,38.0,1124.0,0.0,1154.139120,...,0.0,0.0,6.579749,6.315129,5.832482,6.946683,0.0,0.0,0.0,0.0
2,1,3,3,3226.0,5900.0,1548.0,59.0,1448.0,0.0,1496.832300,...,0.0,0.0,15.497243,16.163633,15.251661,16.716883,0.0,0.0,0.0,0.0
3,1,4,4,6035.0,8804.0,745.0,71.0,621.0,0.0,676.520133,...,0.0,0.0,13.691379,13.019686,13.500400,13.272403,0.0,0.0,0.0,0.0
4,1,5,5,1527.0,2244.0,1219.0,51.0,1175.0,0.0,1193.948265,...,0.0,0.0,0.673830,0.660640,0.689110,0.654166,0.0,0.0,0.0,0.0


In [11]:
df = pd.merge(data, meta, on='ImageNumber')
df['Image_Metadata_WellID'].unique()

array(['A01', 'A02', 'A03', 'A05', 'A06', 'A08', 'B01', 'B02', 'B03',
       'B04', 'B05', 'B06', 'B07', 'C01', 'C02', 'C03', 'C04', 'C05',
       'C06', 'C07', 'D01', 'D02', 'D03', 'D04', 'D05', 'D06', 'E01',
       'E02', 'E03', 'E04', 'E05', 'E06', 'E08', 'F01', 'F02', 'F03',
       'F04', 'F05', 'F06', 'G01', 'G04', 'G05', 'G06', 'G08', 'H01',
       'H03', 'H04', 'H05', 'H06', 'H07', 'H08', 'I04', 'I05', 'I06',
       'I07', 'I08', 'J01', 'J02', 'J05', 'J06', 'J07', 'K01', 'K05',
       'L03', 'L04', 'L05', 'L06', 'L07', 'M01', 'M02', 'M04', 'M05',
       'M07', 'N01', 'N02', 'N04', 'N05', 'N06', 'N07', 'N08', 'O01',
       'O04', 'O05', 'O06', 'O08', 'P01', 'P03', 'P04', 'P05', 'P06'],
      dtype=object)

In [12]:
df.head()

,ImageNumber,ObjectNumber,Cells_Number_Object_Number,Cells_AreaShape_Area,Cells_AreaShape_BoundingBoxArea,Cells_AreaShape_BoundingBoxMaximum_X,Cells_AreaShape_BoundingBoxMaximum_Y,Cells_AreaShape_BoundingBoxMinimum_X,Cells_AreaShape_BoundingBoxMinimum_Y,Cells_AreaShape_Center_X,...,Nuclei_Texture_Variance_Hoe_6_00_256,Nuclei_Texture_Variance_Hoe_6_01_256,Nuclei_Texture_Variance_Hoe_6_02_256,Nuclei_Texture_Variance_Hoe_6_03_256,Nuclei_Texture_Variance_NS4BAfterMath_6_00_256,Nuclei_Texture_Variance_NS4BAfterMath_6_01_256,Nuclei_Texture_Variance_NS4BAfterMath_6_02_256,Nuclei_Texture_Variance_NS4BAfterMath_6_03_256,Image_Metadata_WellID,Image_Metadata_PlateID
0,1,1,1,2565.0,4080.0,622.0,48.0,537.0,0.0,576.887329,...,9.712645,10.273859,9.904125,10.098906,0.0,0.0,0.0,0.0,A01,TimeSeries_20221028_164132
1,1,2,2,1682.0,2090.0,1179.0,38.0,1124.0,0.0,1154.139120,...,6.579749,6.315129,5.832482,6.946683,0.0,0.0,0.0,0.0,A01,TimeSeries_20221028_164132
2,1,3,3,3226.0,5900.0,1548.0,59.0,1448.0,0.0,1496.832300,...,15.497243,16.163633,15.251661,16.716883,0.0,0.0,0.0,0.0,A01,TimeSeries_20221028_164132
3,1,4,4,6035.0,8804.0,745.0,71.0,621.0,0.0,676.520133,...,13.691379,13.019686,13.500400,13.272403,0.0,0.0,0.0,0.0,A01,TimeSeries_20221028_164132
4,1,5,5,1527.0,2244.0,1219.0,51.0,1175.0,0.0,1193.948265,...,0.673830,0.660640,0.689110,0.654166,0.0,0.0,0.0,0.0,A01,TimeSeries_20221028_164132


In [13]:
df['con'] = 'NC'
df.loc[df['Image_Metadata_WellID'].isin(PC), 'con'] = 'PC'

In [14]:
df['label'] = 0
df.loc[df['con']=='PC', 'label'] = 1

In [15]:
df.head()

,ImageNumber,ObjectNumber,Cells_Number_Object_Number,Cells_AreaShape_Area,Cells_AreaShape_BoundingBoxArea,Cells_AreaShape_BoundingBoxMaximum_X,Cells_AreaShape_BoundingBoxMaximum_Y,Cells_AreaShape_BoundingBoxMinimum_X,Cells_AreaShape_BoundingBoxMinimum_Y,Cells_AreaShape_Center_X,...,Nuclei_Texture_Variance_Hoe_6_02_256,Nuclei_Texture_Variance_Hoe_6_03_256,Nuclei_Texture_Variance_NS4BAfterMath_6_00_256,Nuclei_Texture_Variance_NS4BAfterMath_6_01_256,Nuclei_Texture_Variance_NS4BAfterMath_6_02_256,Nuclei_Texture_Variance_NS4BAfterMath_6_03_256,Image_Metadata_WellID,Image_Metadata_PlateID,con,label
0,1,1,1,2565.0,4080.0,622.0,48.0,537.0,0.0,576.887329,...,9.904125,10.098906,0.0,0.0,0.0,0.0,A01,TimeSeries_20221028_164132,NC,0
1,1,2,2,1682.0,2090.0,1179.0,38.0,1124.0,0.0,1154.139120,...,5.832482,6.946683,0.0,0.0,0.0,0.0,A01,TimeSeries_20221028_164132,NC,0
2,1,3,3,3226.0,5900.0,1548.0,59.0,1448.0,0.0,1496.832300,...,15.251661,16.716883,0.0,0.0,0.0,0.0,A01,TimeSeries_20221028_164132,NC,0
3,1,4,4,6035.0,8804.0,745.0,71.0,621.0,0.0,676.520133,...,13.500400,13.272403,0.0,0.0,0.0,0.0,A01,TimeSeries_20221028_164132,NC,0
4,1,5,5,1527.0,2244.0,1219.0,51.0,1175.0,0.0,1193.948265,...,0.689110,0.654166,0.0,0.0,0.0,0.0,A01,TimeSeries_20221028_164132,NC,0


In [16]:
meta_cols = df.columns[df.columns.str.contains(pat='Metadata|ImageNumber|Location|Center|Cells_Number_Object_Number|ObjectNumber', flags=re.IGNORECASE)].tolist()


In [17]:
cols = df.drop(columns=meta_cols).select_dtypes(include='float64').columns.tolist()

In [18]:
data_cols = df[cols].columns[df[cols].columns.str.contains(pat='Epro',flags=re.IGNORECASE)].tolist()

In [19]:
data_cols

['Cells_Intensity_IntegratedIntensityEdge_EproAfterMath',
 'Cells_Intensity_IntegratedIntensity_EproAfterMath',
 'Cells_Intensity_LowerQuartileIntensity_EproAfterMath',
 'Cells_Intensity_MADIntensity_EproAfterMath',
 'Cells_Intensity_MassDisplacement_EproAfterMath',
 'Cells_Intensity_MaxIntensityEdge_EproAfterMath',
 'Cells_Intensity_MaxIntensity_EproAfterMath',
 'Cells_Intensity_MeanIntensityEdge_EproAfterMath',
 'Cells_Intensity_MeanIntensity_EproAfterMath',
 'Cells_Intensity_MedianIntensity_EproAfterMath',
 'Cells_Intensity_MinIntensityEdge_EproAfterMath',
 'Cells_Intensity_MinIntensity_EproAfterMath',
 'Cells_Intensity_StdIntensityEdge_EproAfterMath',
 'Cells_Intensity_StdIntensity_EproAfterMath',
 'Cells_Intensity_UpperQuartileIntensity_EproAfterMath',
 'Cells_RadialDistribution_FracAtD_EproAfterMath_1of6',
 'Cells_RadialDistribution_FracAtD_EproAfterMath_2of6',
 'Cells_RadialDistribution_FracAtD_EproAfterMath_3of6',
 'Cells_RadialDistribution_FracAtD_EproAfterMath_4of6',
 'Cells_

In [20]:
cols = ['ImageNumber', 'Image_Metadata_WellID', 'Image_Metadata_PlateID']+data_cols

In [21]:
df[data_cols] = StandardScaler().fit_transform(df[data_cols])

In [22]:
import xgboost
model = xgboost.XGBRegressor()

In [23]:
from sklearn.model_selection import train_test_split

In [24]:
X = df[data_cols]
y = df['label']

In [25]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [26]:
model.fit(X_train, y_train)

XGBRegressor(base_score=0.5, booster='gbtree', callbacks=None,
             colsample_bylevel=1, colsample_bynode=1, colsample_bytree=1,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, gamma=0, gpu_id=-1,
             grow_policy='depthwise', importance_type=None,
             interaction_constraints='', learning_rate=0.300000012, max_bin=256,
             max_cat_threshold=64, max_cat_to_onehot=4, max_delta_step=0,
             max_depth=6, max_leaves=0, min_child_weight=1, missing=nan,
             monotone_constraints='()', n_estimators=100, n_jobs=0,
             num_parallel_tree=1, predictor='auto', random_state=0, ...)

In [27]:
import pickle

In [28]:
with open('data_cols_reduced', 'wb') as f:
    pickle.dump(data_cols, f)

In [29]:
preds = model.predict(X_test)

In [30]:
from sklearn.metrics import r2_score, mean_squared_error

r2 = r2_score(y_true=y_test, y_pred=preds)
mse = mean_squared_error(y_true=y_test, y_pred=preds)

In [31]:
print("R2", r2)
print("MSE", mse)

R2 0.9437087257648258
MSE 0.013631756455053353


In [32]:
model.save_model('xgb_model_Epro')

In [33]:
del df